# AMEX Enterprise Credit Risk Platform
## Notebook 56 -- Credit Line Management: Validation & Deployment
### Phase 4 . Problem Statement 10: Credit Line Management

CRISP-DM stage: **Evaluation / Deployment**. Depends on Problem 1 Notebooks 01-05, Problem 6 Notebooks 38-40,
and this problem's own Notebooks 54 (real policy) and 55 (real modeling results + worklist).

**What this notebook does:** independently reproduces Notebook 55's real pipeline from scratch (re-scores
every customer with Problem 1's and Problem 6's real persisted models, rebuilds real PD_TREND, re-applies
the SAME already-validated cut values), cross-checks the fresh reproduction against Notebook 55's persisted
`credit_line_worklist.parquet` customer-by-customer, bootstraps 95% confidence intervals on both hard-gating
KPIs (risk_level_monotonicity's ratio and the High-Risk-scoped trend_coherence gap -- see Notebook 54's
2026-09-16 rescope addendum), regenerates `docs/credit_line_deployment_policy.json` and
`src/credit_line_scoring_service.py` with the real, current PD_TREND definition and per-risk-tier trend
cuts, live-self-tests the generated FastAPI service against real customers sampled from each risk tier, and
writes `.env.example` / `requirements-api.txt`.

**Real fix over the version this replaces:** the existing `credit_line_scoring_service.py` and
`credit_line_deployment_policy.json` on disk (dated 2026-08-27T13:17:40Z) were generated by an EARLIER
Notebook 56 run, BEFORE that same day's PD_TREND redefinition took effect -- they still compute
`pd_trend = dynamic_pd - static_pd` (the cross-model definition Notebook 54's addendum documents as
anti-correlated with real outcomes) with a single global trend cut, and report
`recommended_for_production: false`. This run regenerates both with the real, current, 2026-09-16-fixed
definition (`PD_TREND = DYNAMIC_PD - DYNAMIC_PD_EARLY`, per-risk-tier trend cuts, High-Risk-scoped
trend_coherence hard gate) -- a real, necessary API contract change: `/recommend` now takes `dynamic_pd` +
`dynamic_pd_early`, not `static_pd`, to compute trend.

**HYPER note:** `build_trailing_window_store()` and `assign_tiers()` are copied VERBATIM from Notebook 55.
Structure (independent reproduction -> cross-check -> bootstrap CI -> deployment artifacts -> self-test)
follows this platform's established Notebook 68 (Problem 13) Validation & Deployment template.

**WARP note:** same Phase 4 tightened 92%/92% CPU/RAM cap as Notebooks 54/55. Section 5 re-runs the same
double full-streaming pass over the raw CSV as Notebook 55 -- this is this notebook's heaviest step.

Zero-fabrication statement: every number this notebook prints is either computed live (the independent
reproduction, the bootstrap CIs, the live API self-test) or a real value read back from Notebook 55's
already-validated real run -- no results are hardcoded or estimated in advance.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-05, 38-40,
#            54, 55
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 38-40, 54, 55")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB54_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_54_summary.json"
NB55_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_55_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first"),
    (NB54_SUMMARY_PATH, "run 54_credit_line_management_business_understanding.ipynb (Problem 10) first"),
    (NB55_SUMMARY_PATH, "run 55_credit_line_management_modeling.ipynb (Problem 10) first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB54_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB54_SUMMARY = json.load(f)
with open(NB55_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB55_SUMMARY = json.load(f)

POLICY_PATH = Path(NB54_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(f"{POLICY_PATH} not found.\nFix: re-run Notebook 54.")
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    CREDIT_LINE_POLICY = json.load(f)

MODELING_RESULTS_PATH = Path(NB55_SUMMARY["modeling_results_path"])
if not MODELING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"{MODELING_RESULTS_PATH} not found.\nFix: re-run Notebook 55.")
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    NB55_MODELING_RESULTS = json.load(f)

if not NB55_MODELING_RESULTS["all_hard_gates_passed"]:
    print(
        "⚠️  WARNING: Notebook 55's most recent real run did NOT pass all hard gates "
        f"(all_hard_gates_passed={NB55_MODELING_RESULTS['all_hard_gates_passed']}). This notebook still "
        "proceeds -- independent reproduction and validation are meaningful either way -- but every "
        "downstream artifact will honestly carry recommended_for_production=False until a passing "
        "Notebook 55 run exists."
    )

RISK_LEVEL_NAMES = CREDIT_LINE_POLICY["risk_level_names"]
TREND_NAMES = CREDIT_LINE_POLICY["trend_names"]
MIN_STATEMENTS_FOR_DYNAMIC_PD = CREDIT_LINE_POLICY["min_statements_for_dynamic_pd"]
KPI_TARGETS = CREDIT_LINE_POLICY["kpi_targets"]
ACTION_TIER_MATRIX = KPI_TARGETS["action_tier_policy"]["matrix"]
TREND_COHERENCE_HARD_GATE_SCOPE = KPI_TARGETS["trend_coherence"].get("hard_gate_scope", list(RISK_LEVEL_NAMES))

# --- Real cut values persisted by Notebook 55's most recent real run -- this notebook does NOT refit
#     tertile cuts; it re-derives the scores that go INTO those cuts, from scratch, then applies the
#     SAME real cuts Notebook 55 already validated, so this is a genuine independent reproduction of the
#     same pipeline rather than a second, possibly-different fit. ---
RISK_LEVEL_CUT_LOW = NB55_MODELING_RESULTS["risk_level_cut_low"]
RISK_LEVEL_CUT_HIGH = NB55_MODELING_RESULTS["risk_level_cut_high"]
TREND_CUTS_BY_RISK_LEVEL = {
    _rn: (NB55_MODELING_RESULTS["trend_cuts_by_risk_level"][_rn]["low"],
          NB55_MODELING_RESULTS["trend_cuts_by_risk_level"][_rn]["high"])
    for _rn in RISK_LEVEL_NAMES
}

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
_p1_models_subdir = PILLAR_DIRS["model_development"] / "models"
P1_CHAMPION_MODEL_PATH = _p1_models_subdir / (CHAMPION_NAME + ".joblib")
P1_PREPROCESSING_PATH = _p1_models_subdir / "preprocessing_artifacts.joblib"
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_MODEL_PATH = Path(NB40_SUMMARY["model_path"])
P6_PREPROCESSING_PATH = Path(NB40_SUMMARY["preprocessing_path"])
EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

CREDIT_LINE_MODELING_DIR = Path(NB55_MODELING_RESULTS["worklist_path"]).parent
if "credit_line_docs" in PILLAR_DIRS:
    CREDIT_LINE_DOCS_DIR = PILLAR_DIRS["credit_line_docs"]
else:
    CREDIT_LINE_DOCS_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "Problem10_Credit_Line_Management" / "docs"
    )
CREDIT_LINE_DOCS_DIR.mkdir(parents=True, exist_ok=True)
if "credit_line_src" in PILLAR_DIRS:
    CREDIT_LINE_SRC_DIR = PILLAR_DIRS["credit_line_src"]
else:
    CREDIT_LINE_SRC_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "Problem10_Credit_Line_Management" / "src"
    )
CREDIT_LINE_SRC_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DEPLOYMENT_DIR = (
    PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "Problem10_Credit_Line_Management"
    / "validation_deployment"
)
VALIDATION_DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 54's real policy         : {POLICY_PATH}")
print(f"Loaded Notebook 55's real modeling results: {MODELING_RESULTS_PATH}")
print(f"Notebook 55 all_hard_gates_passed         : {NB55_MODELING_RESULTS['all_hard_gates_passed']}")
print(f"Notebook 55 recommended_for_production     : {NB55_MODELING_RESULTS['recommended_for_production']}")
print(f"trend_coherence hard_gate_scope            : {TREND_COHERENCE_HARD_GATE_SCOPE}")
print(f"Docs will be written under  : {CREDIT_LINE_DOCS_DIR}")
print(f"Service will be written under: {CREDIT_LINE_SRC_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 TIGHTENED
#            CAP, SAME 92%/92% AS NOTEBOOKS 54/55)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports (Phase 4 Tightened Cap)")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP))
os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\nFix: pip install " + " ".join(missing)
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs to safely stream the raw "
        f"train_data.csv twice (Section 5). Close other Jupyter kernels, confirm available RAM, and re-run."
    )
print(f"RAM pre-flight check: {_available_ram_gb_at_start:.2f} GB available "
      f"(comfortable margin {_comfortable_available_ram_gb:.2f} GB, hard floor "
      f"{_min_required_available_ram_gb:.2f} GB) -- {'OK' if _available_ram_gb_at_start >= _comfortable_available_ram_gb else 'PROCEEDING, TIGHTER THAN IDEAL'}")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
            f"({WARP_THREAD_COUNT / DETECTED_LOGICAL_CORES:.0%}, Phase 4 tightened cap)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError("Could not find the raw train_data.csv. Checked:\n" +
                             "\n".join(f"  - {c}" for c in _raw_candidates))
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses -- copied verbatim from
    Notebook 55, per this platform's established convention of copying reusable helpers across
    notebooks rather than importing them."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction" / legacy_folder_name
        / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(f"Could not resolve a real, non-trivial {filename}. Checked:\n" +
                             "\n".join(f"  - {c}" for c in _candidates))


TRAIN_FULL_ENGINEERED_PATH = Path(NB04_SUMMARY["output_files"]["train_full_engineered.parquet"])
if not TRAIN_FULL_ENGINEERED_PATH.exists():
    raise FileNotFoundError(f"{TRAIN_FULL_ENGINEERED_PATH} not found.\nFix: re-run Notebook 04.")
TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
WORKLIST_PATH = Path(NB55_MODELING_RESULTS["worklist_path"])
if not WORKLIST_PATH.exists():
    raise FileNotFoundError(f"{WORKLIST_PATH} not found.\nFix: re-run Notebook 55.")

print(f"Raw train_data.csv       : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv     : {RAW_TRAIN_LABELS_PATH}")
print(f"train_split.csv          : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv           : {TEST_SPLIT_PATH}")
print(f"Notebook 55's real worklist (to be cross-checked): {WORKLIST_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: REUSABLE FUNCTIONS -- REUSED VERBATIM FROM NOTEBOOK 55
# =============================================================================
_section("SECTION 4: Reusable Functions -- Reused Verbatim From Notebook 55")


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int, k: int = 0) -> "pl.DataFrame":
    """Copied verbatim from Notebook 55 (itself adapted from Notebook 39, Problem 6), per this
    platform's established convention of copying reusable feature-engineering logic across notebooks
    rather than importing it. Streams csv_path and returns one aggregated row per customer_ID,
    restricted to a real W-statement window ending k statements before the most recent (by real S_2
    date order)."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c) for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(
            (pl.col("_row_idx") >= (pl.col("_n_statements") - w - k))
            & (pl.col("_row_idx") < (pl.col("_n_statements") - k))
        )
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )

    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None).var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}")).otherwise(None).alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))

    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]
    return grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID").collect(engine="streaming")


def assign_tiers(df: "pl.DataFrame") -> "pl.DataFrame":
    """Copied verbatim from Notebook 55's 2026-09-16-fixed _assign_tiers: RISK_LEVEL from DYNAMIC_PD,
    then TREND from the cut pair that matches each customer's OWN RISK_LEVEL (per-tier trend cuts)."""
    _risk_expr = (
        pl.when(pl.col("DYNAMIC_PD") <= RISK_LEVEL_CUT_LOW).then(pl.lit(RISK_LEVEL_NAMES[0]))
        .when(pl.col("DYNAMIC_PD") <= RISK_LEVEL_CUT_HIGH).then(pl.lit(RISK_LEVEL_NAMES[1]))
        .otherwise(pl.lit(RISK_LEVEL_NAMES[2])).alias("RISK_LEVEL")
    )
    df = df.with_columns([_risk_expr])
    _trend_expr = pl.lit(None, dtype=pl.Utf8)
    for _risk_name in reversed(RISK_LEVEL_NAMES):
        _lo, _hi = TREND_CUTS_BY_RISK_LEVEL[_risk_name]
        _this_tier_trend = (
            pl.when(pl.col("PD_TREND") <= _lo).then(pl.lit(TREND_NAMES[0]))
            .when(pl.col("PD_TREND") <= _hi).then(pl.lit(TREND_NAMES[1]))
            .otherwise(pl.lit(TREND_NAMES[2]))
        )
        _trend_expr = pl.when(pl.col("RISK_LEVEL") == _risk_name).then(_this_tier_trend).otherwise(_trend_expr)
    return df.with_columns(_trend_expr.alias("TREND"))


print("build_trailing_window_store() and assign_tiers() defined, copied verbatim from Notebook 55.")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: INDEPENDENT REPRODUCTION OF NOTEBOOK 55'S REAL PIPELINE
# =============================================================================
_section("SECTION 5: Independent Reproduction of Notebook 55's Real Pipeline")

# --- Real, from-scratch re-derivation: re-scores every customer with Problem 1's and Problem 6's real
#     persisted models, rebuilds real PD_TREND, and re-assigns real RISK_LEVEL/TREND using the SAME
#     already-validated cut values Notebook 55 persisted (Section 1) -- this notebook does not refit
#     cuts, it verifies that the SAME pipeline, run independently, reproduces the SAME real customer-
#     level outcomes Notebook 55 persisted. ---
P1_PREPROCESSING = joblib.load(P1_PREPROCESSING_PATH)
P1_ALL_FEATURE_COLS = P1_PREPROCESSING["all_feature_cols"]
P1_CATEGORICAL_COLS = P1_PREPROCESSING["categorical_encode_cols"]
P1_NUMERIC_COLS = P1_PREPROCESSING["numeric_feature_cols"]
P1_LABEL_ENCODERS = P1_PREPROCESSING["label_encoders"]
P1_FEATURE_MEDIANS = P1_PREPROCESSING["feature_medians"]
P1_CHAMPION_USES_SCALED = CHAMPION_NAME == "logistic_regression"
if P1_CHAMPION_USES_SCALED:
    P1_SCALER = P1_PREPROCESSING["scaler"]

print(f"Reading Problem 1's real whole-history engineered feature matrix: {TRAIN_FULL_ENGINEERED_PATH}")
_t0 = time.time()
_engineered_cols = ["customer_ID"] + P1_ALL_FEATURE_COLS
_train_full_eng = pl.read_parquet(TRAIN_FULL_ENGINEERED_PATH, columns=_engineered_cols)
_cat_exprs = []
for _c in P1_CATEGORICAL_COLS:
    _classes = P1_LABEL_ENCODERS[_c]["classes"]
    _mapping = {cat: idx for idx, cat in enumerate(_classes)}
    _default = _mapping.get("__missing__", -1)
    _cat_exprs.append(
        pl.col(_c).cast(pl.Utf8).fill_null("__missing__").replace_strict(_mapping, default=_default)
        .cast(pl.Float32).alias(_c)
    )
_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P1_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c) for _c in P1_NUMERIC_COLS
]
_train_full_eng = _train_full_eng.with_columns(_cat_exprs + _num_exprs)
STATIC_PD_DF = _train_full_eng.select(["customer_ID"] + P1_ALL_FEATURE_COLS)
_X_static = STATIC_PD_DF.select(P1_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _train_full_eng
gc.collect()
P1_CHAMPION_MODEL = joblib.load(P1_CHAMPION_MODEL_PATH)
if P1_CHAMPION_USES_SCALED:
    _X_static = (_X_static - np.asarray(P1_SCALER["mean"], dtype=np.float32)) / np.asarray(
        P1_SCALER["std"], dtype=np.float32)
_static_proba = P1_CHAMPION_MODEL.predict_proba(_X_static)[:, 1]
STATIC_PD_DF = STATIC_PD_DF.select("customer_ID").with_columns(pl.Series("STATIC_PD", _static_proba, dtype=pl.Float64))
del _X_static, _static_proba
gc.collect()
print(f"Re-scored {STATIC_PD_DF.height:,} customers with Problem 1's real champion model in {time.time() - _t0:.1f}s. "
      f"RSS: {_rss_gb():.2f} GB")

P6_PREPROCESSING = joblib.load(P6_PREPROCESSING_PATH)
P6_FEATURE_MEDIANS = P6_PREPROCESSING["feature_medians"]
P6_ALL_FEATURE_COLS = P6_PREPROCESSING["all_feature_cols"]
P6_BASE_FEATURE_COLUMNS = P6_PREPROCESSING["base_feature_columns"]
P6_MODEL = joblib.load(P6_MODEL_PATH)

print(f"\nRe-building the real trailing-{P6_WINNING_W}-statement feature store (current window). "
      f"RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
_p6_store = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W)
_p6_store = _p6_store.filter(pl.col("_actual_window_len") == P6_WINNING_W)
_p6_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c) for _c in P6_ALL_FEATURE_COLS
]
_p6_store = _p6_store.with_columns(_p6_num_exprs)
DYNAMIC_PD_DF = _p6_store.select(["customer_ID"] + P6_ALL_FEATURE_COLS)
_X_dynamic = DYNAMIC_PD_DF.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _p6_store
gc.collect()
_dynamic_proba = P6_MODEL.predict_proba(_X_dynamic)[:, 1]
DYNAMIC_PD_DF = DYNAMIC_PD_DF.select("customer_ID").with_columns(pl.Series("DYNAMIC_PD", _dynamic_proba, dtype=pl.Float64))
del _X_dynamic, _dynamic_proba
gc.collect()
print(f"Re-built and re-scored current window in {time.time() - _t0:.1f}s: {DYNAMIC_PD_DF.height:,} customers. "
      f"RSS: {_rss_gb():.2f} GB")

print(f"\nRe-building the real, EARLIER, non-overlapping {P6_WINNING_W}-statement window. "
      f"RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
_p6_store_early = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W, k=P6_WINNING_W)
_p6_store_early = _p6_store_early.filter(pl.col("_actual_window_len") == P6_WINNING_W)
_p6_early_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c) for _c in P6_ALL_FEATURE_COLS
]
_p6_store_early = _p6_store_early.with_columns(_p6_early_num_exprs)
DYNAMIC_PD_EARLY_DF = _p6_store_early.select(["customer_ID"] + P6_ALL_FEATURE_COLS)
_X_dynamic_early = DYNAMIC_PD_EARLY_DF.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _p6_store_early
gc.collect()
_dynamic_early_proba = P6_MODEL.predict_proba(_X_dynamic_early)[:, 1]
DYNAMIC_PD_EARLY_DF = DYNAMIC_PD_EARLY_DF.select("customer_ID").with_columns(
    pl.Series("DYNAMIC_PD_EARLY", _dynamic_early_proba, dtype=pl.Float64))
del _X_dynamic_early, _dynamic_early_proba
gc.collect()
print(f"Re-built and re-scored earlier window in {time.time() - _t0:.1f}s: {DYNAMIC_PD_EARLY_DF.height:,} customers. "
      f"RSS: {_rss_gb():.2f} GB")

TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
REPRO_DF = (
    STATIC_PD_DF.join(DYNAMIC_PD_DF, on="customer_ID", how="inner")
    .join(DYNAMIC_PD_EARLY_DF, on="customer_ID", how="inner")
    .join(TARGET_DF, on="customer_ID", how="inner")
    .with_columns((pl.col("DYNAMIC_PD") - pl.col("DYNAMIC_PD_EARLY")).alias("PD_TREND"))
)
REPRO_DF = assign_tiers(REPRO_DF)

TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
REPRO_HOLDOUT_DF = REPRO_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")
print(f"\nIndependently reproduced eligible population: {REPRO_DF.height:,} customers "
      f"(Notebook 55's real run: {NB55_MODELING_RESULTS['eligible_population']:,})")
print(f"Independently reproduced holdout population : {REPRO_HOLDOUT_DF.height:,} customers "
      f"(Notebook 55's real run: {NB55_MODELING_RESULTS['holdout_split_population']:,})")

_repro_auc = float(roc_auc_score(REPRO_HOLDOUT_DF["target"].to_numpy(), REPRO_HOLDOUT_DF["DYNAMIC_PD"].to_numpy()))
print(f"Independently reproduced DYNAMIC_PD ROC-AUC (real, HOLDOUT): {_repro_auc:.4f} "
      f"(Notebook 55's real run: {NB55_MODELING_RESULTS['dynamic_pd_roc_auc']:.4f})")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: CROSS-CHECK THE PERSISTED WORKLIST AGAINST A FRESH REPRODUCTION
# =============================================================================
_section("SECTION 6: Cross-Check the Persisted Worklist Against a Fresh Reproduction")

_persisted = pl.read_parquet(WORKLIST_PATH).select(
    ["customer_ID", "DYNAMIC_PD", "PD_TREND", "RISK_LEVEL", "TREND", "ACTION"]
).rename({
    "DYNAMIC_PD": "DYNAMIC_PD_persisted", "PD_TREND": "PD_TREND_persisted",
    "RISK_LEVEL": "RISK_LEVEL_persisted", "TREND": "TREND_persisted", "ACTION": "ACTION_persisted",
})
_action_map_repro = {(c["risk_level"], c["trend"]): c["action"] for c in ACTION_TIER_MATRIX}
REPRO_DF = REPRO_DF.with_columns(
    pl.struct(["RISK_LEVEL", "TREND"]).map_elements(
        lambda s: _action_map_repro[(s["RISK_LEVEL"], s["TREND"])], return_dtype=pl.Utf8
    ).alias("ACTION")
)

_compare = REPRO_DF.join(_persisted, on="customer_ID", how="inner")
_n_compared = _compare.height
_dynamic_pd_match = _compare.filter((pl.col("DYNAMIC_PD") - pl.col("DYNAMIC_PD_persisted")).abs() < 1e-6).height
_pd_trend_match = _compare.filter((pl.col("PD_TREND") - pl.col("PD_TREND_persisted")).abs() < 1e-6).height
_risk_level_match = _compare.filter(pl.col("RISK_LEVEL") == pl.col("RISK_LEVEL_persisted")).height
_trend_match = _compare.filter(pl.col("TREND") == pl.col("TREND_persisted")).height
_action_match = _compare.filter(pl.col("ACTION") == pl.col("ACTION_persisted")).height

print(f"Customers matched between fresh reproduction and Notebook 55's persisted worklist: {_n_compared:,} "
      f"of {REPRO_DF.height:,} reproduced / {_persisted.height:,} persisted")
print(f"  DYNAMIC_PD  matches (tol 1e-6): {_dynamic_pd_match:,} / {_n_compared:,} "
      f"({100.0 * _dynamic_pd_match / _n_compared:.4f}%)")
print(f"  PD_TREND    matches (tol 1e-6): {_pd_trend_match:,} / {_n_compared:,} "
      f"({100.0 * _pd_trend_match / _n_compared:.4f}%)")
print(f"  RISK_LEVEL  matches (exact)   : {_risk_level_match:,} / {_n_compared:,} "
      f"({100.0 * _risk_level_match / _n_compared:.4f}%)")
print(f"  TREND       matches (exact)   : {_trend_match:,} / {_n_compared:,} "
      f"({100.0 * _trend_match / _n_compared:.4f}%)")
print(f"  ACTION      matches (exact)   : {_action_match:,} / {_n_compared:,} "
      f"({100.0 * _action_match / _n_compared:.4f}%)")

WORKLIST_VERIFIED = bool(
    _n_compared == REPRO_DF.height == _persisted.height
    and _dynamic_pd_match == _n_compared and _pd_trend_match == _n_compared
    and _risk_level_match == _n_compared and _trend_match == _n_compared and _action_match == _n_compared
)
print(f"\nworklist_verified: {WORKLIST_VERIFIED} -- independent reproduction "
      f"{'matches' if WORKLIST_VERIFIED else 'does NOT fully match'} Notebook 55's persisted real output.")
if not WORKLIST_VERIFIED:
    print(
        "HONEST FINDING: a mismatch between independent reproduction and the persisted worklist means "
        "either non-determinism somewhere in the real pipeline or a real discrepancy -- this is reported "
        "plainly, not hidden, and worklist_verified=False propagates to the deployment policy below."
    )
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: BOOTSTRAP CONFIDENCE INTERVALS -- BOTH HARD-GATING KPIS
# =============================================================================
_section("SECTION 7: Bootstrap Confidence Intervals -- Both Hard-Gating KPIs")

# --- Same bootstrap convention as Notebook 68 (Problem 13): 200 resamples, RANDOM_SEED=42, real
#     resampling-with-replacement of the real HOLDOUT split, 95% CI via the 2.5/97.5 percentiles. Two
#     KPIs bootstrapped here: risk_level_monotonicity's top/bottom default-rate ratio, and the High-Risk
#     tier's trend_coherence gap (Trending Worse minus Trending Better default rate) -- the only tier
#     the 2026-09-16 rescope actually hard-gates on (see Notebook 54's trend_coherence addendum). ---
_y = REPRO_HOLDOUT_DF["target"].to_numpy()
_risk = REPRO_HOLDOUT_DF["RISK_LEVEL"].to_numpy()
_trend = REPRO_HOLDOUT_DF["TREND"].to_numpy()
_n_holdout = len(_y)
_rng = np.random.default_rng(RANDOM_SEED)
_n_boot = 200

_boot_ratios = np.empty(_n_boot, dtype=np.float64)
_boot_high_risk_gaps = np.empty(_n_boot, dtype=np.float64)
_low_name, _, _high_name = RISK_LEVEL_NAMES
_better_name, _, _worse_name = TREND_NAMES

for _i in range(_n_boot):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _yb, _riskb, _trendb = _y[_idx], _risk[_idx], _trend[_idx]

    _low_mask = _riskb == _low_name
    _high_mask = _riskb == _high_name
    _low_rate = _yb[_low_mask].mean() if _low_mask.sum() > 0 else np.nan
    _high_rate = _yb[_high_mask].mean() if _high_mask.sum() > 0 else np.nan
    _boot_ratios[_i] = (_high_rate / _low_rate) if (_low_rate and _low_rate > 0) else np.nan

    _hr_better_mask = _high_mask & (_trendb == _better_name)
    _hr_worse_mask = _high_mask & (_trendb == _worse_name)
    _hr_better_rate = _yb[_hr_better_mask].mean() if _hr_better_mask.sum() > 0 else np.nan
    _hr_worse_rate = _yb[_hr_worse_mask].mean() if _hr_worse_mask.sum() > 0 else np.nan
    _boot_high_risk_gaps[_i] = _hr_worse_rate - _hr_better_rate

_ratios_valid = _boot_ratios[~np.isnan(_boot_ratios)]
RISK_LEVEL_RATIO_CI = [float(np.percentile(_ratios_valid, 2.5)), float(np.percentile(_ratios_valid, 97.5))] \
    if len(_ratios_valid) > 0 else [float("nan"), float("nan")]
_gaps_valid = _boot_high_risk_gaps[~np.isnan(_boot_high_risk_gaps)]
HIGH_RISK_TREND_GAP_CI = [float(np.percentile(_gaps_valid, 2.5)), float(np.percentile(_gaps_valid, 97.5))] \
    if len(_gaps_valid) > 0 else [float("nan"), float("nan")]

print(f"Bootstrap 95% CI on risk_level_monotonicity top/bottom ratio ({len(_ratios_valid)} valid resamples "
      f"of {_n_boot}): [{RISK_LEVEL_RATIO_CI[0]:.2f}, {RISK_LEVEL_RATIO_CI[1]:.2f}]")
print(f"Bootstrap 95% CI on High Risk trend_coherence gap (Worse - Better default rate, "
      f"{len(_gaps_valid)} valid resamples of {_n_boot}): "
      f"[{HIGH_RISK_TREND_GAP_CI[0]:.4f}, {HIGH_RISK_TREND_GAP_CI[1]:.4f}]")

_ratio_threshold = KPI_TARGETS["risk_level_monotonicity"]["min_default_rate_ratio_top_to_bottom_tier"]
RISK_LEVEL_MONOTONICITY_CI_PASSED = bool(RISK_LEVEL_RATIO_CI[0] >= _ratio_threshold)
HIGH_RISK_TREND_COHERENCE_CI_PASSED = bool(HIGH_RISK_TREND_GAP_CI[0] > 0.0)
MEETS_KPI_WITH_CI = bool(RISK_LEVEL_MONOTONICITY_CI_PASSED and HIGH_RISK_TREND_COHERENCE_CI_PASSED)

print(f"\nrisk_level_monotonicity CI lower bound >= {_ratio_threshold}: {RISK_LEVEL_MONOTONICITY_CI_PASSED}")
print(f"High Risk trend_coherence CI lower bound > 0 (entire CI stays positive): "
      f"{HIGH_RISK_TREND_COHERENCE_CI_PASSED}")
print(f"meets_kpi_with_ci (both, real bootstrap, not just the point estimate): {MEETS_KPI_WITH_CI}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 8: Honest Limitation -- Deployment Scope & Assumptions")

print(
    "HONEST LIMITATIONS carried into the deployment policy and API docs (not hidden):\n"
    "  1. TREND only differentiates the recommended ACTION within the High Risk tier (2026-09-16 rescope). "
    "Low Risk and Medium Risk always receive the same action regardless of trend -- see Notebook 54's "
    "action_tier_policy description for the real data behind this.\n"
    "  2. PD_TREND requires a real DYNAMIC_PD_EARLY score (Problem 6's model applied to an immediately-"
    "preceding, non-overlapping window) -- a customer needs >= 2x the trailing window width "
    f"({2 * P6_WINNING_W} real statements) to have one. Customers below that bar are NOT eligible for "
    "this API's /recommend endpoint; STATIC_PD alone (Problem 1) is the fallback for those accounts, "
    "outside this notebook's scope.\n"
    "  3. The action-tier matrix (Notebook 54) is an explicit ASSUMPTION-based business-rule policy layer, "
    "not a fitted treatment-response model -- no real limit-change/outcome data exists in this dataset to "
    "fit one against.\n"
    f"  4. worklist_verified={WORKLIST_VERIFIED} and recommended_for_production reflects Notebook 55's "
    "real, most recent run plus this notebook's own bootstrap CI check -- both must hold for this API to "
    "report recommended_for_production=True."
)
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: PERSIST DEPLOYMENT POLICY ARTIFACT
# =============================================================================
_section("SECTION 9: Persist Deployment Policy Artifact")

# --- Regenerates docs/credit_line_deployment_policy.json. The version this notebook replaces was
#     generated 2026-08-27 by an EARLIER Notebook 56 run, BEFORE that same day's PD_TREND redefinition
#     took effect -- it still used pd_trend = dynamic_pd - static_pd (the cross-model definition Notebook
#     54's addendum documents as anti-correlated with real outcomes) and a single global trend cut. This
#     run replaces it with the real, current, 2026-09-16-fixed definition: PD_TREND = DYNAMIC_PD -
#     DYNAMIC_PD_EARLY, per-risk-tier trend cuts, and the High-Risk-scoped trend_coherence hard gate. ---
RECOMMENDED_FOR_PRODUCTION_FINAL = bool(
    NB55_MODELING_RESULTS["recommended_for_production"] and WORKLIST_VERIFIED and MEETS_KPI_WITH_CI
)

DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 10 -- Credit Line Management (Deployment Policy)",
    "supersedes": "The 2026-08-27T13:17:40Z version of this file, which used the pre-redefinition "
                   "pd_trend = dynamic_pd - static_pd formula and a single global trend cut -- both "
                   "superseded, see Notebook 54's trend_coherence and pd_trend_definition addenda.",
    "pd_trend_definition": NB55_MODELING_RESULTS["pd_trend_definition"],
    "risk_level_names": RISK_LEVEL_NAMES,
    "trend_names": TREND_NAMES,
    "risk_level_cut_low": RISK_LEVEL_CUT_LOW,
    "risk_level_cut_high": RISK_LEVEL_CUT_HIGH,
    "trend_cuts_by_risk_level": {
        _rn: {"low": TREND_CUTS_BY_RISK_LEVEL[_rn][0], "high": TREND_CUTS_BY_RISK_LEVEL[_rn][1]}
        for _rn in RISK_LEVEL_NAMES
    },
    "min_statements_for_dynamic_pd": MIN_STATEMENTS_FOR_DYNAMIC_PD,
    "min_statements_for_pd_trend": 2 * P6_WINNING_W,
    "action_tier_matrix": ACTION_TIER_MATRIX,
    "trend_coherence_hard_gate_scope": TREND_COHERENCE_HARD_GATE_SCOPE,
    "reported_dynamic_pd_roc_auc": NB55_MODELING_RESULTS["dynamic_pd_roc_auc"],
    "reproduced_dynamic_pd_roc_auc": _repro_auc,
    "reproduction_passed": bool(abs(_repro_auc - NB55_MODELING_RESULTS["dynamic_pd_roc_auc"]) < 1e-4),
    "worklist_verified": WORKLIST_VERIFIED,
    "risk_level_ratio": NB55_MODELING_RESULTS["kpi_results"]["risk_level_monotonicity"]["top_to_bottom_ratio"],
    "risk_level_ratio_ci_95": RISK_LEVEL_RATIO_CI,
    "high_risk_trend_coherence_gap_ci_95": HIGH_RISK_TREND_GAP_CI,
    "risk_level_monotonicity_ci_passed": RISK_LEVEL_MONOTONICITY_CI_PASSED,
    "high_risk_trend_coherence_ci_passed": HIGH_RISK_TREND_COHERENCE_CI_PASSED,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION_FINAL,
    "ead_per_account_usd": EAD_PER_ACCOUNT_USD,
    "lgd_assumption": LGD_ASSUMPTION,
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = CREDIT_LINE_DOCS_DIR / "credit_line_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(DEPLOYMENT_POLICY, f, indent=2)
print(f"Wrote: {deployment_policy_path}")
print(f"recommended_for_production (final, this notebook): {RECOMMENDED_FOR_PRODUCTION_FINAL}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: GENERATE credit_line_scoring_service.py -- REAL, RUNNABLE
# =============================================================================
_section("SECTION 10: Generate credit_line_scoring_service.py -- Real, Runnable")

# --- Regenerates src/credit_line_scoring_service.py. Real, necessary API contract change from the
#     2026-08-27 version: PD_TREND is no longer dynamic_pd - static_pd, so /recommend now takes
#     dynamic_pd + dynamic_pd_early (not static_pd) as its trend inputs, and trend classification uses
#     the per-risk-tier cut pair, not one global cut. static_pd is still accepted and echoed back for
#     reference/reporting, but no longer drives the trend calculation. ---
SERVICE_CODE = f'''# AMEX Enterprise Credit Risk Platform -- Credit Line Management Recommendation API.
# Auto-generated by 56_credit_line_management_validation_deployment.ipynb (regenerated 2026-09-16).
# Composes a customer's real dynamic PD (Problem 6, current window) and real dynamic PD_EARLY (Problem 6,
# immediately-preceding window) into a real PD_TREND, then a risk-level x trend classification and a real
# credit-line action recommendation. static_pd (Problem 1) is accepted and echoed for reference only -- it
# no longer drives the trend calculation (see Notebook 54's 2026-08-27 pd_trend_definition addendum for why
# the original cross-model definition was replaced).
# Every endpoint except /health requires a valid X-API-Key header (see .env.example).
# Run with:
#     uvicorn credit_line_scoring_service:app --host 0.0.0.0 --port 8010
import json
import logging
import os
import secrets
from pathlib import Path
from typing import List, Optional

from fastapi import Depends, FastAPI, HTTPException, Security
from fastapi.security import APIKeyHeader
from pydantic import BaseModel

_auth_logger = logging.getLogger(__name__ + ".auth")
_DEV_DEFAULT_API_KEY = "dev-only-CHANGE-ME-before-deploying"
_api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def _configured_api_key() -> str:
    key = os.environ.get("API_KEY")
    if not key:
        _auth_logger.warning(
            "API_KEY is not set -- falling back to the published dev-only default. Set API_KEY before "
            "deploying this service anywhere reachable by anyone but you."
        )
        return _DEV_DEFAULT_API_KEY
    return key


def require_api_key(presented: str = Security(_api_key_header)) -> str:
    expected = _configured_api_key()
    if not presented or not secrets.compare_digest(presented, expected):
        raise HTTPException(status_code=401, detail="Missing or invalid X-API-Key header.")
    return presented


POLICY_PATH = Path(os.environ.get("AMEX_P10_POLICY_PATH", r"{deployment_policy_path}"))
with open(POLICY_PATH, "r", encoding="utf-8") as _f:
    _POLICY = json.load(_f)

RISK_LEVEL_NAMES = _POLICY["risk_level_names"]
TREND_NAMES = _POLICY["trend_names"]
RISK_LEVEL_CUT_LOW = _POLICY["risk_level_cut_low"]
RISK_LEVEL_CUT_HIGH = _POLICY["risk_level_cut_high"]
TREND_CUTS_BY_RISK_LEVEL = _POLICY["trend_cuts_by_risk_level"]
MIN_STATEMENTS_FOR_PD_TREND = _POLICY["min_statements_for_pd_trend"]
ACTION_MAP = {{(c["risk_level"], c["trend"]): c for c in _POLICY["action_tier_matrix"]}}
TREND_COHERENCE_HARD_GATE_SCOPE = _POLICY["trend_coherence_hard_gate_scope"]
RECOMMENDED_FOR_PRODUCTION = _POLICY["recommended_for_production"]
PD_TREND_DEFINITION = _POLICY["pd_trend_definition"]


class RecommendRequest(BaseModel):
    customer_id: Optional[str] = None
    static_pd: Optional[float] = None
    dynamic_pd: float
    dynamic_pd_early: float


class RecommendResponse(BaseModel):
    customer_id: Optional[str] = None
    static_pd: Optional[float] = None
    dynamic_pd: float
    dynamic_pd_early: float
    pd_trend: float
    risk_level: str
    trend: str
    trend_is_hard_gated: bool
    action: str
    rationale: str
    reasoning: List[str] = []
    recommended_for_production: bool = RECOMMENDED_FOR_PRODUCTION


def _assign_risk_level(dynamic_pd: float) -> str:
    if dynamic_pd <= RISK_LEVEL_CUT_LOW:
        return RISK_LEVEL_NAMES[0]
    if dynamic_pd <= RISK_LEVEL_CUT_HIGH:
        return RISK_LEVEL_NAMES[1]
    return RISK_LEVEL_NAMES[2]


def _assign_trend(risk_level: str, pd_trend: float) -> str:
    _lo = TREND_CUTS_BY_RISK_LEVEL[risk_level]["low"]
    _hi = TREND_CUTS_BY_RISK_LEVEL[risk_level]["high"]
    if pd_trend <= _lo:
        return TREND_NAMES[0]
    if pd_trend <= _hi:
        return TREND_NAMES[1]
    return TREND_NAMES[2]


app = FastAPI(
    title="AMEX Enterprise Credit Risk Platform -- Credit Line Management Recommendation API",
    description="Composes real current + earlier dynamic PD scores into a per-risk-tier PD_TREND, a "
                "risk-level x trend classification, and a real credit-line action recommendation. Every "
                "endpoint except /health requires a valid X-API-Key header.",
    version="2.0.0",
)


@app.get("/health")
def health():
    return {{"status": "ok"}}


@app.get("/policy-info", dependencies=[Depends(require_api_key)])
def policy_info():
    return {{
        "risk_level_names": RISK_LEVEL_NAMES,
        "trend_names": TREND_NAMES,
        "risk_level_cuts": [RISK_LEVEL_CUT_LOW, RISK_LEVEL_CUT_HIGH],
        "trend_cuts_by_risk_level": TREND_CUTS_BY_RISK_LEVEL,
        "action_tier_matrix": _POLICY["action_tier_matrix"],
        "trend_coherence_hard_gate_scope": TREND_COHERENCE_HARD_GATE_SCOPE,
        "min_statements_for_pd_trend": MIN_STATEMENTS_FOR_PD_TREND,
        "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
        "pd_trend_definition": PD_TREND_DEFINITION,
    }}


@app.post("/recommend", response_model=RecommendResponse, dependencies=[Depends(require_api_key)])
def recommend(request: RecommendRequest):
    if not (0.0 <= request.dynamic_pd <= 1.0) or not (0.0 <= request.dynamic_pd_early <= 1.0):
        raise HTTPException(status_code=422, detail="dynamic_pd and dynamic_pd_early must both be in [0, 1].")
    if request.static_pd is not None and not (0.0 <= request.static_pd <= 1.0):
        raise HTTPException(status_code=422, detail="static_pd, if provided, must be in [0, 1].")
    pd_trend = request.dynamic_pd - request.dynamic_pd_early
    risk_level = _assign_risk_level(request.dynamic_pd)
    trend = _assign_trend(risk_level, pd_trend)
    cell = ACTION_MAP.get((risk_level, trend))
    if cell is None:
        raise HTTPException(status_code=500, detail=f"No action defined for ({{risk_level}}, {{trend}}).")
    trend_is_hard_gated = risk_level in TREND_COHERENCE_HARD_GATE_SCOPE
    _lo = TREND_CUTS_BY_RISK_LEVEL[risk_level]["low"]
    _hi = TREND_CUTS_BY_RISK_LEVEL[risk_level]["high"]
    reasoning = [
        f"dynamic_pd={{request.dynamic_pd:.4f}} -> {{risk_level}} "
        f"(cuts: <= {{RISK_LEVEL_CUT_LOW:.4f}} Low, <= {{RISK_LEVEL_CUT_HIGH:.4f}} Medium, else High)",
        f"pd_trend={{pd_trend:.4f}} (dynamic_pd - dynamic_pd_early, {{risk_level}}-tier cuts: "
        f"<= {{_lo:.6f}} Better, <= {{_hi:.6f}} Stable, else Worse) -> {{trend}}"
        + (" [hard-gated, validated signal]" if trend_is_hard_gated
           else " [informational only -- not used to differentiate action in this tier, 2026-09-16 rescope]"),
        f"({{risk_level}}, {{trend}}) -> {{cell['action']}}",
    ]
    return RecommendResponse(
        customer_id=request.customer_id, static_pd=request.static_pd, dynamic_pd=request.dynamic_pd,
        dynamic_pd_early=request.dynamic_pd_early, pd_trend=pd_trend, risk_level=risk_level, trend=trend,
        trend_is_hard_gated=trend_is_hard_gated, action=cell["action"], rationale=cell["rationale"],
        reasoning=reasoning,
    )
'''

service_path = CREDIT_LINE_SRC_DIR / "credit_line_scoring_service.py"
with open(service_path, "w", encoding="utf-8") as f:
    f.write(SERVICE_CODE)
print(f"Wrote: {service_path} ({len(SERVICE_CODE):,} chars)")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#             REAL CUSTOMERS
# =============================================================================
_section("SECTION 11: Live Self-Test -- Import the Generated Service & Drive It With Real Customers")

sys.path.insert(0, str(CREDIT_LINE_SRC_DIR))
os.environ["AMEX_P10_POLICY_PATH"] = str(deployment_policy_path)
for _mod_name in list(sys.modules):
    if _mod_name == "credit_line_scoring_service":
        del sys.modules[_mod_name]

_self_test_ok = True
try:
    import credit_line_scoring_service as _svc
    from fastapi.testclient import TestClient
    _client = TestClient(_svc.app)

    _health = _client.get("/health")
    print(f"GET /health -> {_health.status_code} {_health.json()}")
    _self_test_ok &= _health.status_code == 200

    _api_key = os.environ.get("API_KEY", _svc._DEV_DEFAULT_API_KEY)
    _headers = {"X-API-Key": _api_key}

    _policy_resp = _client.get("/policy-info", headers=_headers)
    print(f"GET /policy-info -> {_policy_resp.status_code}, "
          f"recommended_for_production={_policy_resp.json().get('recommended_for_production')}")
    _self_test_ok &= _policy_resp.status_code == 200

    # --- Drive /recommend with 3 REAL customers, one sampled from each real risk tier of the
    #     independently reproduced population (Section 5), not synthetic inputs. ---
    _sample_rows = []
    for _risk_name in RISK_LEVEL_NAMES:
        _row = REPRO_DF.filter(pl.col("RISK_LEVEL") == _risk_name).head(1)
        if _row.height > 0:
            _sample_rows.append(_row.to_dicts()[0])

    for _row in _sample_rows:
        _payload = {
            "customer_id": _row["customer_ID"], "static_pd": _row["STATIC_PD"],
            "dynamic_pd": _row["DYNAMIC_PD"], "dynamic_pd_early": _row["DYNAMIC_PD_EARLY"],
        }
        _resp = _client.post("/recommend", json=_payload, headers=_headers)
        _body = _resp.json()
        print(f"POST /recommend customer_ID={_row['customer_ID']}: {_resp.status_code} -> "
              f"risk_level={_body.get('risk_level')}, trend={_body.get('trend')}, "
              f"action={_body.get('action')}, trend_is_hard_gated={_body.get('trend_is_hard_gated')}")
        _self_test_ok &= _resp.status_code == 200
        _self_test_ok &= _body.get("risk_level") == _row["RISK_LEVEL"]
        _self_test_ok &= _body.get("trend") == _row["TREND"]
        _self_test_ok &= _body.get("action") == _row["ACTION"]

    _bad_resp = _client.post("/recommend", json={"dynamic_pd": 1.5, "dynamic_pd_early": 0.1}, headers=_headers)
    print(f"POST /recommend with out-of-range dynamic_pd -> {_bad_resp.status_code} (expect 422)")
    _self_test_ok &= _bad_resp.status_code == 422

    _noauth_resp = _client.get("/policy-info")
    print(f"GET /policy-info with no API key -> {_noauth_resp.status_code} (expect 401)")
    _self_test_ok &= _noauth_resp.status_code == 401

except Exception as _e:
    print(f"❌ Self-test raised an exception: {_e!r}")
    _self_test_ok = False

SELF_TEST_PASSED = bool(_self_test_ok)
print(f"\nSELF_TEST_PASSED: {SELF_TEST_PASSED}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 12: Generate .env.example & requirements-api.txt")

ENV_EXAMPLE = f"""# Copy to .env and fill in before deploying credit_line_scoring_service.py anywhere reachable
# by anyone but you.
API_KEY=replace-with-a-real-secret-before-deploying
AMEX_P10_POLICY_PATH={deployment_policy_path}
"""
env_path = CREDIT_LINE_SRC_DIR / ".env.example"
with open(env_path, "w", encoding="utf-8") as f:
    f.write(ENV_EXAMPLE)
print(f"Wrote: {env_path}")

REQUIREMENTS_API = "fastapi\nuvicorn[standard]\npydantic\n"
requirements_path = CREDIT_LINE_SRC_DIR / "requirements-api.txt"
with open(requirements_path, "w", encoding="utf-8") as f:
    f.write(REQUIREMENTS_API)
print(f"Wrote: {requirements_path}")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 13: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Deployment policy file was written", deployment_policy_path.exists())
_all_checks_passed &= _check("Service file was written", service_path.exists())
_all_checks_passed &= _check(".env.example was written", env_path.exists())
_all_checks_passed &= _check("requirements-api.txt was written", requirements_path.exists())
_all_checks_passed &= _check(
    "Independent reproduction's DYNAMIC_PD ROC-AUC matches Notebook 55's real run (tol 1e-4)",
    abs(_repro_auc - NB55_MODELING_RESULTS["dynamic_pd_roc_auc"]) < 1e-4,
    detail=f"repro={_repro_auc:.6f}, NB55={NB55_MODELING_RESULTS['dynamic_pd_roc_auc']:.6f}",
)
_all_checks_passed &= _check("Persisted worklist matches independent reproduction (worklist_verified)",
                              WORKLIST_VERIFIED)
_all_checks_passed &= _check("Bootstrap CI for risk_level_monotonicity is internally consistent (low <= high)",
                              RISK_LEVEL_RATIO_CI[0] <= RISK_LEVEL_RATIO_CI[1])
_all_checks_passed &= _check("Bootstrap CI for High Risk trend_coherence gap is internally consistent (low <= high)",
                              HIGH_RISK_TREND_GAP_CI[0] <= HIGH_RISK_TREND_GAP_CI[1])
_all_checks_passed &= _check("Live self-test of the generated API passed", SELF_TEST_PASSED)
_all_checks_passed &= _check(
    "recommended_for_production is internally consistent across policy JSON and this notebook's own checks",
    DEPLOYMENT_POLICY["recommended_for_production"] == RECOMMENDED_FOR_PRODUCTION_FINAL,
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 13 complete -- all checks passed.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 56 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 14: Write Notebook 56 Summary Artifact")

NB56_SUMMARY = {
    "notebook": "56_credit_line_management_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "deployment_policy_path": str(deployment_policy_path),
    "service_path": str(service_path),
    "worklist_verified": WORKLIST_VERIFIED,
    "reproduced_dynamic_pd_roc_auc": _repro_auc,
    "risk_level_ratio_ci_95": RISK_LEVEL_RATIO_CI,
    "high_risk_trend_coherence_gap_ci_95": HIGH_RISK_TREND_GAP_CI,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "self_test_passed": SELF_TEST_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION_FINAL,
    "warp_thread_count": WARP_THREAD_COUNT,
    "random_seed": RANDOM_SEED,
}
NB56_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_56_summary.json"
with open(NB56_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB56_SUMMARY, f, indent=2)
print(f"Wrote: {NB56_SUMMARY_PATH}")

_section("NOTEBOOK 56 COMPLETE")
print(f"worklist_verified                    : {WORKLIST_VERIFIED}")
print(f"meets_kpi_with_ci (bootstrap, real)   : {MEETS_KPI_WITH_CI}")
print(f"self_test_passed                      : {SELF_TEST_PASSED}")
print(f"RECOMMENDED_FOR_PRODUCTION (final)    : {RECOMMENDED_FOR_PRODUCTION_FINAL}")
print(f"Deployment policy: {deployment_policy_path}")
print(f"Service          : {service_path}")
print(
    "\nNext: 57_credit_line_management_financial_impact_reporting_packaging.ipynb -- computes the real "
    "financial impact (loss prevention from High Risk Freeze/Review actions, revenue from Low/Medium Risk "
    "Increase actions, net of review cost), generates smart suggestions, rebuilds charts, and packages the "
    "Word/Excel/HTML reports -- the ones already on disk from 2026-08-27 are stale relative to this fix."
)